# 2026 高教社杯 C 题 · 问题 3 求解（修复版）

**含日内预报更新的微网四阶段购电策略**

本版统一修复以下口径：

1. 所有新增参数只用 **2025年1月** 选择；2—12月仅用于部署和事后评价。
2. 0:00、6:00、12:00、18:00 每个阶段都优化未来 **严格48小时**。
3. 附件3发布后的完整24小时预报按绝对目标时刻使用，跨日部分不再丢弃。
4. 48小时全窗口均加入历史残差经验分位数安全余量。
5. $[Q_{\alpha_{lo}},Q_{\alpha_{hi}}]$ 只称为单时段边际分析启发的阶段级规则；参数由一月滚动验证确定。
6. 若当日剩余时段全部落在带内，则该阶段明确跳过调整；否则裁剪目标并重解含储能耦合的线性规划。
7. 问题2基准由最终同口径代码重算；所有费用由当前运行结果生成，不再手工填写。

运行只需要官方附件1、附件2、附件3及 `result3.xlsx` 模板。代码会在本目录和已知官方附件目录中自动查找。

## 1 信息边界与参数来源

- **选择期**：1月11—31日。1月1—10日只作滚动预测预热；选择期内每一天只读取该日决策时点以前已完成的真值。
- **部署期**：2月1—12月31日。模型形式、候选集及最终参数在2月1日前冻结；后续真值只在其揭晓后进入滚动历史。
- **继承问题2的固定设置**：同类日EW、最近8天、衰减0.6、残差窗口14天、储能参数及实时结算规则。
- **问题3新增并由一月选择**：附件3时间对齐、小时到10分钟降尺度方法、0:00计划分位数和不调整带。
- **初始化**：2月1日0:00取 $S_0=6000$ kWh，之后SOC连续传递，不每日重置。

> 一月用于选择，不把2—12月费用反向用于调参。

In [1]:
from __future__ import annotations

from functools import lru_cache
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.optimize import linprog
from scipy.sparse import csr_matrix, lil_matrix

# ---------- constants ----------
T, DT = 144, 1 / 6
START = 31
SELECT_DAYS = range(10, START)
ISSUE = [0, 6, 12, 18]
STAGE_SLOT = [0, 36, 72, 108]
ETA, R_MAX, SMIN, SMAX = 0.9, 5000 / 6, 1200.0, 10800.0
PEN, UP, DN = 5.0, 1.5, 0.5
EW_N, EW_DECAY, RESID_LOOK, FUSER_LOOK = 8, 0.6, 14, 14
DAY0 = pd.Timestamp("2025-01-01")


def resolve_file(name: str, candidates: list[str]) -> Path:
    for raw in candidates:
        p = Path(raw)
        if p.exists():
            return p
    raise FileNotFoundError(f"找不到 {name}；已检查：{candidates}")


DATA1 = resolve_file("附件1.xlsx", [
    "附件1.xlsx", r"C:\Users\12055\Desktop\2026 国赛\附件1.xlsx",
    r"C:\Users\12055\Desktop\Hermes\2026国赛C题\official\extracted\C题\附件\附件1.xlsx",
])
DATA2 = resolve_file("附件2.xlsx", [
    "附件2.xlsx", r"C:\Users\12055\Desktop\2026 国赛\附件2.xlsx",
    r"C:\Users\12055\Desktop\Hermes\2026国赛C题\official\extracted\C题\附件\附件2.xlsx",
])
DATA3 = resolve_file("附件3.xlsx", [
    "附件3.xlsx", r"C:\Users\12055\Desktop\2026 国赛\附件3.xlsx",
    r"C:\Users\12055\Desktop\Hermes\2026国赛C题\official\extracted\C题\附件\附件3.xlsx",
])
RESULT3_TEMPLATE = resolve_file("result3.xlsx", [
    "result3.xlsx", r"C:\Users\12055\Desktop\2026 国赛\附件5\result3.xlsx",
    r"C:\Users\12055\Desktop\Hermes\2026国赛C题\official\extracted\C题\附件\附件5\result3.xlsx",
])

price = pd.read_excel(DATA1)["电价"].to_numpy(float)
Ld = pd.read_excel(DATA2, sheet_name="小区负载")
L = Ld.iloc[:, 1:].to_numpy(float)
Gv = pd.read_excel(DATA2, sheet_name="光伏发电实际功率").iloc[:, 1:].to_numpy(float)
dates = pd.DatetimeIndex(pd.to_datetime(Ld.iloc[:, 0]))
ND = len(dates)
assert L.shape == Gv.shape == (365, T)
assert len(price) == T

F = pd.read_excel(DATA3)
F["日期"] = pd.to_datetime(F["日期"].ffill())
F["h0"] = F["预报时刻"].astype(str).str.split(":").str[0].astype(int)
FC = np.full((ND, 4, 25), np.nan)
for _, row in F.iterrows():
    d = (row["日期"] - DAY0).days
    if 0 <= d < ND:
        k = ISSUE.index(int(row["h0"]))
        for h in range(1, 25):
            FC[d, k, h] = float(row[f"预报{h}小时"])

Lflat, Gflat = L.ravel(), Gv.ravel()


def low_type(target_day: int) -> bool:
    return (DAY0 + pd.Timedelta(days=int(target_day))).dayofweek in (4, 5)


@lru_cache(None)
def ew_load_day(cutoff: int, target_day: int) -> np.ndarray:
    idx = [j for j in range(cutoff - 1, -1, -1) if low_type(j) == low_type(target_day)][:EW_N]
    w = EW_DECAY ** np.arange(len(idx)); w /= w.sum()
    return np.sum(w[:, None] * L[idx], axis=0)


@lru_cache(None)
def ew_pv_day(cutoff: int, target_day: int) -> np.ndarray:
    idx = list(range(cutoff - 1, max(-1, cutoff - EW_N - 1), -1))
    w = EW_DECAY ** np.arange(len(idx)); w /= w.sum()
    pred = np.sum(w[:, None] * Gv[idx], axis=0)
    pred[Gv[:cutoff].max(axis=0) <= 0] = 0
    return np.maximum(pred, 0)


def ew_abs(kind: str, cutoff: int, abs_slots: np.ndarray) -> np.ndarray:
    out = np.empty(len(abs_slots), float)
    for td in np.unique(abs_slots // T):
        m = abs_slots // T == td
        day_pred = ew_load_day(cutoff, int(td)) if kind == "load" else ew_pv_day(cutoff, int(td))
        out[m] = day_pred[abs_slots[m] % T]
    return out


def downscale_24h(d: int, k: int, method: str = "linear") -> tuple[np.ndarray, np.ndarray]:
    """保留发布后完整24小时；返回绝对目标时段及10分钟预报。"""
    release = d * T + STAGE_SLOT[k]
    x = np.arange(release, release + T)  # release + T 明确跨日保留
    anchors = release + 6 * np.arange(1, 25) - 1
    vals = FC[d, k, 1:25].astype(float)
    prev = Gflat[release - 1] if release > 0 else 0.0
    if method == "linear":
        out = np.interp(x, np.r_[release - 1, anchors], np.r_[prev, vals])
    elif method == "hold":
        out = np.repeat(vals, 6)
    elif method == "shape":
        base = ew_abs("pv", d, x)
        out = np.interp(x, np.r_[release - 1, anchors], np.r_[prev, vals])
        for pos, value in zip(anchors, vals):
            m = (x >= pos - 5) & (x <= pos)
            seg = base[m]
            if len(seg) and seg[-1] > 1e-6:
                out[m] = value * seg / seg[-1]
    else:
        raise ValueError(method)
    return x, np.maximum(out, 0)


def january_alignment_table() -> pd.DataFrame:
    rows = []
    hourly = Gv.reshape(ND, 24, 6).mean(2).ravel()
    for mode in ("小时均值", "小时末瞬时", "小时初瞬时"):
        for off in (0, -1):
            err = []
            for d in range(START):
                for k, h0 in enumerate(ISSUE):
                    for h in range(1, 25):
                        hour_abs = d * 24 + h0 + h + off
                        if mode == "小时均值":
                            if 0 <= hour_abs < START * 24:
                                err.append(FC[d, k, h] - hourly[hour_abs])
                        else:
                            slot_abs = hour_abs * 6 - (1 if mode == "小时末瞬时" else 0)
                            if 0 <= slot_abs < START * T:
                                err.append(FC[d, k, h] - Gflat[slot_abs])
            rows.append({"对齐方式": mode, "偏移": off, "一月MAE/kW": np.mean(np.abs(err))})
    return pd.DataFrame(rows).sort_values("一月MAE/kW").reset_index(drop=True)


def january_downscale_table() -> pd.DataFrame:
    rows = []
    for method in ("hold", "linear", "shape"):
        err = []
        for d in range(8, START):
            for k in range(4):
                pos, pred = downscale_24h(d, k, method)
                m = pos < START * T
                act = Gflat[pos[m]]; p = pred[m]
                daylight = (act > 50) | (p > 50)
                if daylight.any(): err.extend(p[daylight] - act[daylight])
        e = np.asarray(err)
        rows.append({"方法": method, "一月MAE/kW": np.abs(e).mean(), "一月RMSE/kW": np.sqrt(np.mean(e**2))})
    return pd.DataFrame(rows).sort_values("一月MAE/kW").reset_index(drop=True)


# 方法只由一月选择；在后续期冻结。
ALIGNMENT_TABLE = january_alignment_table()
DOWNSCALE_TABLE = january_downscale_table()
SELECTED_DOWNSCALE = str(DOWNSCALE_TABLE.iloc[0]["方法"])
FC10 = np.full((ND, 4, T), np.nan)
for d in range(ND):
    for k in range(4):
        FC10[d, k] = downscale_24h(d, k, SELECTED_DOWNSCALE)[1]


@lru_cache(None)
def point_forecast48(d: int, k: int, use_attachment: bool = True) -> np.ndarray:
    """决策时刻起严格48小时净负荷点预测，只用d日前真值与当时已发布预报。"""
    release = d * T + STAGE_SLOT[k]
    abs_slots = np.arange(release, release + 2 * T)
    load_hat = ew_abs("load", d, abs_slots)
    pv_ew = ew_abs("pv", d, abs_slots)
    pv_hat = pv_ew.copy()
    if use_attachment:
        for r in range(T):
            target = release + r
            if target >= len(Gflat) + T:
                continue
            efc, eew = [], []
            for j in range(d - 1, max(7, d - FUSER_LOOK - 3), -1):
                hist_target = j * T + STAGE_SLOT[k] + r
                if hist_target >= release or hist_target >= len(Gflat):
                    continue
                f = FC10[j, k, r]
                if not np.isfinite(f):
                    continue
                actual = Gflat[hist_target]
                ew0 = ew_abs("pv", j, np.array([hist_target]))[0]
                if actual > 50 or f > 50:
                    efc.append(f - actual); eew.append(ew0 - actual)
                if len(efc) >= FUSER_LOOK:
                    break
            fcur = FC10[d, k, r]
            if len(efc) < 5:
                pv_hat[r] = 0.5 * fcur + 0.5 * pv_ew[r]
            else:
                efc = np.asarray(efc); eew = np.asarray(eew)
                bias = efc.mean(); vfc = max(efc.var(), 1.0); vew = max(eew.var(), 1.0)
                fcur -= bias
                pv_hat[r] = ((1 / vfc) * fcur + (1 / vew) * pv_ew[r]) / ((1 / vfc) + (1 / vew))
    return (load_hat - np.maximum(pv_hat, 0)) * DT


@lru_cache(None)
def residual_samples48(d: int, k: int, look: int = RESID_LOOK) -> np.ndarray:
    """每个相对时段取最近look个、在当前发布时刻前已揭晓的残差。"""
    release = d * T + STAGE_SLOT[k]
    samples = np.full((look, 2 * T), np.nan)
    for r in range(2 * T):
        vals = []
        for j in range(d - 1, 7, -1):
            target = j * T + STAGE_SLOT[k] + r
            if target >= release or target >= len(Gflat):
                continue
            vals.append((Lflat[target] - Gflat[target]) * DT - point_forecast48(j, k)[r])
            if len(vals) == look:
                break
        if vals:
            samples[:len(vals), r] = vals
    return samples


def rho48(d: int, k: int, alpha: float, look: int = RESID_LOOK) -> np.ndarray:
    """严格48小时全窗的经验分位数余量；今天、明天及跨日部分均不遗漏。"""
    s = residual_samples48(d, k, look)
    out = np.nanquantile(s, alpha, axis=0)
    return np.nan_to_num(out, nan=0.0)


def solve_stage48(d: int, k: int, target48: np.ndarray, S0: float,
                  qplan: np.ndarray | None = None) -> np.ndarray:
    """每个阶段严格优化未来48小时，只执行到当日24:00。"""
    H = 2 * T
    target48 = np.asarray(target48, float)
    if target48.shape != (H,):
        raise ValueError("target48必须恰为288个10分钟时段")
    t0 = STAGE_SLOT[k]; n_exec = T - t0
    abs_slots = d * T + t0 + np.arange(H)
    ph = price[abs_slots % T]
    nv = 4 * H + (n_exec if qplan is not None else 0)
    iq, ic, iu, iS = (np.arange(H) + j * H for j in range(4))
    cobj = np.zeros(nv)
    if qplan is None:
        cobj[iq] = ph
    else:
        cobj[iq[n_exec:]] = ph[n_exec:]
        iy = 4 * H + np.arange(n_exec); cobj[iy] = 1.0
    lb = np.zeros(nv); ub = np.full(nv, np.inf)
    lb[iS] = SMIN; ub[iS] = SMAX
    if qplan is not None:
        lb[iy] = -np.inf
    rows, cols, vals, rhs = [], [], [], []; rr = 0; tt = np.arange(H)
    rows += [rr + tt, rr + tt, rr + tt]; cols += [iq, iu, ic]
    vals += [-np.ones(H), -np.ones(H), np.ones(H)]; rhs.append(-target48); rr += H
    rows += [rr + tt, rr + tt]; cols += [ic, iu]
    vals += [np.ones(H), np.ones(H)]; rhs.append(np.full(H, R_MAX)); rr += H
    if qplan is not None:
        pd0 = price[t0:]
        for coef in (DN, UP):
            r0 = rr + np.arange(n_exec)
            rows += [r0, r0]; cols += [iq[:n_exec], iy]
            vals += [coef * pd0, -np.ones(n_exec)]
            rhs.append(coef * pd0 * qplan[t0:]); rr += n_exec
    A_ub = csr_matrix((np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))), shape=(rr, nv))
    er = [tt, tt[1:], tt, tt]; ec = [iS, iS[:-1], ic, iu]
    ev = [np.ones(H), -np.ones(H - 1), -ETA * np.ones(H), np.ones(H) / ETA]
    A_eq = csr_matrix((np.concatenate(ev), (np.concatenate(er), np.concatenate(ec))), shape=(H, nv))
    b_eq = np.zeros(H); b_eq[0] = S0
    res = linprog(cobj, A_ub=A_ub, b_ub=np.concatenate(rhs), A_eq=A_eq, b_eq=b_eq,
                  bounds=np.c_[lb, ub], method="highs")
    if not res.success:
        raise RuntimeError(f"{dates[min(d, ND-1)].date()} 阶段{k} LP失败：{res.message}")
    return res.x[iq[:n_exec]]


def settle(q: np.ndarray, net_act: np.ndarray, S0: float):
    n = len(q); c = np.zeros(n); u = np.zeros(n); e = np.zeros(n); w = np.zeros(n); S = np.zeros(n); s = float(S0)
    for t in range(n):
        gap = net_act[t] - q[t]
        if gap > 0:
            u[t] = min(gap, R_MAX, (s - SMIN) * ETA); e[t] = gap - u[t]; s -= u[t] / ETA
        else:
            c[t] = min(-gap, R_MAX, (SMAX - s) / ETA); w[t] = -gap - c[t]; s += ETA * c[t]
        S[t] = s
    return c, u, e, w, S


def state_at(q: np.ndarray, net_act: np.ndarray, S0: float, t0: int) -> float:
    if t0 == 0:
        return S0
    return float(settle(q[:t0], net_act[:t0], S0)[4][-1])


def simulate(stages=(1, 2, 3), A0=0.7, ALO=0.7, AHI=0.9,
             S_init=6000.0, days=None, use_attachment=True):
    days = list(range(START, ND) if days is None else days)
    if not days or days != list(range(days[0], days[-1] + 1)):
        raise ValueError("days必须为非空连续日期范围")
    rec = {k: [] for k in ["qplan", "qadj", "c", "u", "e", "w", "S", "S0"]}
    s = float(S_init)
    for d in days:
        q = np.zeros(T)
        pred0 = point_forecast48(d, 0, use_attachment)
        target0 = pred0 + rho48(d, 0, A0)
        q[:] = solve_stage48(d, 0, target0, s)
        qplan = q.copy(); plan_target_day = target0[:T].copy()
        for k in stages:
            t0 = STAGE_SLOT[k]; n_exec = T - t0
            pred = point_forecast48(d, k, use_attachment)
            lo = pred + rho48(d, k, ALO)
            hi = pred + rho48(d, k, AHI)
            # 阶段级启发式：全部剩余时段都在带内则不调整；否则裁剪目标后重解耦合LP。
            base = plan_target_day[t0:]
            inside_band = (base >= lo[:n_exec]) & (base <= hi[:n_exec])
            if np.all(inside_band):
                continue
            target48 = pred + rho48(d, k, A0)  # 明天及整个48小时同样带安全余量
            target48[:n_exec] = np.clip(base, lo[:n_exec], hi[:n_exec])
            s_now = state_at(q, (L[d] - Gv[d]) * DT, s, t0)
            q[t0:] = solve_stage48(d, k, target48, s_now, qplan)
        c, u, e, w, S = settle(q, (L[d] - Gv[d]) * DT, s)
        for key, value in zip(["qplan", "qadj", "c", "u", "e", "w", "S"], [qplan, q, c, u, e, w, S]):
            rec[key].append(value)
        rec["S0"].append(s); s = float(S[-1])
    return {k: np.asarray(v) for k, v in rec.items()}


def cost(rec: dict[str, np.ndarray]) -> dict[str, float]:
    P = price[None, :]; qp, qa, e = rec["qplan"], rec["qadj"], rec["e"]
    plan = np.sum(qp * P); up = np.sum(UP * P * np.maximum(qa - qp, 0))
    dn = np.sum(DN * P * np.maximum(qp - qa, 0)); emg = np.sum(PEN * P * e)
    return {"计划购电费/万元": plan / 1e4, "调增费/万元": up / 1e4,
            "调减退费/万元": -dn / 1e4, "紧急购电费/万元": emg / 1e4,
            "总费用/万元": (plan + up - dn + emg) / 1e4,
            "紧急购电量/万kWh": e.sum() / 1e4,
            "发生紧急购电天数": int(np.sum(e.sum(axis=1) > 1e-6))}


def select_parameters_january(a0_values=(0.6, 0.7, 0.8, 0.9),
                              bands=((0.5, 0.99), (0.6, 0.95), (0.7, 0.9), (0.75, 0.85))):
    rows = []
    for a0 in a0_values:
        for lo, hi in bands:
            rec = simulate(A0=a0, ALO=lo, AHI=hi, days=SELECT_DAYS)
            rows.append({"A0": a0, "ALO": lo, "AHI": hi, **cost(rec)})
    table = pd.DataFrame(rows).sort_values(["总费用/万元", "A0", "ALO"]).reset_index(drop=True)
    best = table.iloc[0]
    return table, {"A0": float(best.A0), "ALO": float(best.ALO), "AHI": float(best.AHI)}


# ---------- Q2 exact baseline: copied specification from final Q2 ----------
def solve_plan_lp(load_energy, pv_energy, price48, s0):
    h = 2 * T; nv = 6 * h
    eu, es, qu, qs, v, soc = [np.arange(h) + k * h for k in range(6)]
    cobj = np.zeros(nv); cobj[qu] = price48; cobj[qs] = price48
    bounds = [(0, None)] * (5 * h) + [(SMIN, SMAX)] * h
    aub = lil_matrix((2 * h, nv)); bub = np.r_[pv_energy, np.full(h, R_MAX)]
    for t in range(h):
        aub[t, eu[t]] = 1; aub[t, es[t]] = 1
        aub[h + t, es[t]] = 1; aub[h + t, qs[t]] = 1; aub[h + t, v[t]] = 1
    aeq = lil_matrix((2 * h, nv)); beq = np.r_[load_energy, s0, np.zeros(h - 1)]
    for t in range(h):
        aeq[t, eu[t]] = 1; aeq[t, qu[t]] = 1; aeq[t, v[t]] = 1
        r = h + t; aeq[r, soc[t]] = 1; aeq[r, es[t]] = -ETA; aeq[r, qs[t]] = -ETA; aeq[r, v[t]] = 1 / ETA
        if t > 0: aeq[r, soc[t - 1]] = -1
    res = linprog(cobj, A_ub=aub.tocsr(), b_ub=bub, A_eq=aeq.tocsr(), b_eq=beq,
                  bounds=bounds, method="highs")
    if not res.success: raise RuntimeError(res.message)
    return res.x[qu][:T] + res.x[qs][:T]


def simulate_q2_baseline(alpha=0.8, look=14, S_init=6000.0):
    hist = []
    for d in range(10, START):
        pred = (ew_load_day(d, d) - ew_pv_day(d, d)) * DT
        hist.append((L[d] - Gv[d]) * DT - pred)
    qrows, erows, s = [], [], float(S_init)
    price48 = np.r_[price, price]
    for d in range(START, ND):
        rho = np.quantile(np.asarray(hist[-look:]), alpha, axis=0)
        l48 = np.r_[ew_load_day(d, d), ew_load_day(d, d + 1)] * DT + np.r_[rho, rho]
        p48 = np.r_[ew_pv_day(d, d), ew_pv_day(d, d + 1)] * DT
        q = solve_plan_lp(l48, p48, price48, s)
        _, _, e, _, S = settle(q, (L[d] - Gv[d]) * DT, s); s = float(S[-1])
        pred = (ew_load_day(d, d) - ew_pv_day(d, d)) * DT
        hist.append((L[d] - Gv[d]) * DT - pred); hist = hist[-60:]
        qrows.append(q); erows.append(e)
    rec = {"qplan": np.asarray(qrows), "qadj": np.asarray(qrows), "e": np.asarray(erows)}
    return rec, cost(rec)


In [2]:
print(f"负荷/光伏：{ND}天 × {T}时段；部署期 {dates[START].date()}—{dates[-1].date()}，共{ND-START}天")
print("读取文件：", DATA1, DATA2, DATA3, sep="\n- ")
print("\n一月时间对齐比较：")
display(ALIGNMENT_TABLE)
print("\n一月降尺度比较：")
display(DOWNSCALE_TABLE)
print(f"冻结选择：小时末瞬时、偏移0；降尺度={SELECTED_DOWNSCALE}")

负荷/光伏：365天 × 144时段；部署期 2025-02-01—2025-12-31，共334天
读取文件：
- C:\Users\12055\Desktop\2026 国赛\附件1.xlsx
- C:\Users\12055\Desktop\2026 国赛\附件2.xlsx
- C:\Users\12055\Desktop\2026 国赛\附件3.xlsx

一月时间对齐比较：


,对齐方式,偏移,一月MAE/kW
0,小时末瞬时,0,129.882864
1,小时初瞬时,0,180.344948
2,小时均值,-1,256.972987
3,小时均值,0,355.921456
4,小时初瞬时,-1,447.322392
5,小时末瞬时,-1,525.335555



一月降尺度比较：


,方法,一月MAE/kW,一月RMSE/kW
0,linear,291.741889,442.809665
1,shape,344.788305,512.103548
2,hold,600.129869,781.930265


冻结选择：小时末瞬时、偏移0；降尺度=linear


## 2 一月参数选择

调整费用的单时段边际条件可以解释候选带的来源：调增边界对应 $Q_{0.7}$，调减边界对应 $Q_{0.9}$。但储能会造成跨时段耦合，因此该带不能直接宣称为动态模型的严格解。

同理，后续存在调整机会时，0:00计划分位数不能仅由对称边际费用推出。下面在预先冻结的一月选择期内，联合比较候选计划分位数与候选带；2—12月不参与选择。

In [3]:
selection_table, PARAMS = select_parameters_january()
print("一月联合选择结果（按总费用排序，前10项）：")
display(selection_table.head(10))
print("冻结参数：", PARAMS)
assert set(SELECT_DAYS) == set(range(10, START))
assert selection_table.shape[0] == 16

一月联合选择结果（按总费用排序，前10项）：


,A0,ALO,AHI,计划购电费/万元,调增费/万元,调减退费/万元,紧急购电费/万元,总费用/万元,紧急购电量/万kWh,发生紧急购电天数
0,0.6,0.70,0.90,103.040495,3.467192,-0.475587,0.593233,106.625332,0.110551,11
1,0.7,0.75,0.85,104.688793,2.281990,-0.682454,0.383750,106.672079,0.072197,10
2,0.7,0.70,0.90,104.758716,1.642107,-0.640933,0.995952,106.755843,0.167993,10
3,0.6,0.75,0.85,102.995248,3.963976,-0.507190,0.389548,106.841582,0.073347,11
4,0.7,0.60,0.95,104.764292,1.329478,-0.582703,1.332188,106.843255,0.231421,11
5,0.8,0.75,0.85,106.371166,0.986846,-0.894468,0.522252,106.985796,0.088577,10
6,0.7,0.50,0.99,104.770232,1.071113,-0.544963,1.737864,107.034246,0.306898,11
7,0.8,0.70,0.90,106.370878,0.899021,-0.830089,0.610700,107.050510,0.104069,10
8,0.6,0.60,0.95,103.149781,2.173115,-0.434596,2.187977,107.076278,0.377765,15
9,0.8,0.60,0.95,106.370606,0.726258,-0.747987,0.812140,107.161017,0.141748,10


冻结参数： {'A0': 0.6, 'ALO': 0.7, 'AHI': 0.9}


## 3 严格48小时机会约束与阶段调整

每个决策时刻都构造从当前时刻起的288个10分钟时段。设净负荷点预测为 $\hat n_{d,k,r}$，只使用当前发布时刻前已揭晓的历史预测残差，得到经验分位数余量 $\rho_{d,k,r}(\alpha)$，则全窗口目标为

$$
b_{d,k,r}=\hat n_{d,k,r}+\rho_{d,k,r}(\alpha),\qquad r=0,\ldots,287.
$$

因此，今天、明天以及日内阶段跨到后天的部分都使用同一规则，不再对次日只放点预测。

阶段级调整规则如下：

- 当日剩余全部目标都处于最新带内：保持当前购电计划，不调用调整LP；
- 只要存在带外目标：将当日剩余目标裁剪到分位数带边界，再求解严格48小时LP；
- 由于储能耦合，重解时带内时段也可能随整体可行性改变，因此本文不把逐时段带规则写成严格动态最优。

In [4]:
import time
run_start = time.time()
Q2_REC, Q2_COST = simulate_q2_baseline()
Q3_NO = simulate(stages=(), **PARAMS)
FIN = simulate(stages=(1, 2, 3), **PARAMS)
Q3_NO_COST, Q3_COST = cost(Q3_NO), cost(FIN)
summary = pd.DataFrame([Q2_COST, Q3_NO_COST, Q3_COST],
                       index=["问题2最终方案（同口径重算）", "问题3仅0:00计划", "问题3四阶段方案"])
display(summary)
q2 = Q2_COST["总费用/万元"]; q3 = Q3_COST["总费用/万元"]; no = Q3_NO_COST["总费用/万元"]
print(f"问题3相对问题2：节省 {q2-q3:.4f} 万元，下降 {(q2-q3)/q2:.3%}")
print(f"三次日内调整相对仅0:00计划：节省 {no-q3:.4f} 万元，下降 {(no-q3)/no:.3%}")
print(f"运行用时：{time.time()-run_start:.1f}秒")

,计划购电费/万元,调增费/万元,调减退费/万元,紧急购电费/万元,总费用/万元,紧急购电量/万kWh,发生紧急购电天数
问题2最终方案（同口径重算）,1316.412144,0.000000,-0.000000,47.020814,1363.432958,7.745867,134
问题3仅0:00计划,1248.321296,0.000000,-0.000000,143.857041,1392.178337,23.283929,256
问题3四阶段方案,1245.967919,54.801386,-7.040036,21.216826,1314.946095,3.824971,162


问题3相对问题2：节省 48.4869 万元，下降 3.556%
三次日内调整相对仅0:00计划：节省 77.2322 万元，下降 5.548%
运行用时：76.3秒


## 4 结果解释边界

- “是否需要日内预报”由四阶段方案与仅0:00方案的同口径费用差回答。
- 不比较“哪个时刻最值钱”：按6:00→12:00→18:00顺序增加阶段得到的是顺序相关的嵌套边际，不能当作各时刻独立贡献。
- 附件3误差随提前时长总体增大，但不声称逐小时单调。
- 预测方案及费用均由当前代码运行生成；不保留旧版手填结果。

In [5]:
# 信息边界与实现断言
assert len(SELECT_DAYS) == 21 and max(SELECT_DAYS) < START
assert point_forecast48(START, 3).shape == (288,)
assert rho48(START, 3, PARAMS["A0"]).shape == (288,)
# 18:00预报的后6小时落在次日，确认没有被自然日边界截断
release = START * T + STAGE_SLOT[3]
assert np.array_equal(np.arange(release, release + T)[-36:] // T,
                      np.full(36, START + 1))
# 费用和SOC基本可行性
assert np.all((FIN["S"] >= SMIN - 1e-6) & (FIN["S"] <= SMAX + 1e-6))
assert np.allclose(FIN["S0"][1:], FIN["S"][:-1, -1])
assert np.isfinite(summary.select_dtypes("number").to_numpy()).all()
print("验证通过：一月选参、完整跨日预报、严格48小时余量、SOC连续和费用动态生成。")

验证通过：一月选参、完整跨日预报、严格48小时余量、SOC连续和费用动态生成。


## 5 指定日期结果

In [6]:
fd = dates[START:]
qp, qa, C_, U_, E_, S_, S0_ = (FIN[k] for k in ["qplan", "qadj", "c", "u", "e", "S", "S0"])
P = price[None, :]
fee = ((qp * P).sum(1) + (UP * P * np.maximum(qa-qp, 0)).sum(1)
       - (DN * P * np.maximum(qp-qa, 0)).sum(1) + (PEN * E_ * P).sum(1))
SPEC = pd.to_datetime(["2025-03-20", "2025-06-21", "2025-09-23", "2025-12-21"])
rows = []
for day in SPEC:
    i = int(np.where(fd == day)[0][0])
    rows.append({"日期": day.date(), "计划购电量/kWh": qp[i].sum(),
                 "调整后购电量/kWh": qa[i].sum(), "紧急购电量/kWh": E_[i].sum(),
                 "日初SOC/kWh": S0_[i], "日末SOC/kWh": S_[i,-1], "购电费/元": fee[i]})
display(pd.DataFrame(rows).set_index("日期"))

,计划购电量/kWh,调整后购电量/kWh,紧急购电量/kWh,日初SOC/kWh,日末SOC/kWh,购电费/元
日期,,,,,,
2025-03-20,60164.715828,61421.927744,325.403355,8790.823186,4469.863222,40469.627408
2025-06-21,39397.901472,39841.827898,0.000000,3542.803398,9299.077621,22235.227705
2025-09-23,66514.369321,67217.104853,165.998662,8650.798981,8573.993716,43886.519633
2025-12-21,94628.368354,95816.423958,48.116651,8776.026004,8659.060816,62214.436631


## 6 导出 `result3_filled.xlsx`

In [7]:
from openpyxl import load_workbook
wb = load_workbook(RESULT3_TEMPLATE)
fee_plan = (qp * P).sum(1)
fee_adj = fee_plan + (UP * P * np.maximum(qa-qp, 0)).sum(1) - (DN * P * np.maximum(qp-qa, 0)).sum(1)
for sheet, arr, fees in (("计划购电量", qp, fee_plan), ("调整购电量", qa, fee_adj)):
    ws = wb[sheet]
    for i in range(len(fd)):
        for j in range(T): ws.cell(i+2, j+2).value = float(max(arr[i,j], 0))
        ws.cell(i+2, T+2).value = float(arr[i].sum()); ws.cell(i+2, T+3).value = float(fees[i])
ws = wb["充放电量"]; ws.delete_rows(2, ws.max_row); row = 2
for i in range(len(fd)):
    for g in range(6):
        sl = slice(g*24, (g+1)*24)
        if g == 0:
            ws.cell(row,1).value = fd[i].strftime("%Y/%m/%d")
            ws.cell(row,5).value = "0:00"; ws.cell(row,6).value = float(S0_[i])
        elif g == 1:
            ws.cell(row,5).value = "24:00"; ws.cell(row,6).value = float(S_[i,-1])
        ws.cell(row,2).value = f"{g*4}:00-{(g+1)*4}:00"
        ws.cell(row,3).value = float(C_[i,sl].sum()); ws.cell(row,4).value = float(U_[i,sl].sum()); row += 1
ws = wb["紧急购电量"]; ws.delete_rows(2, ws.max_row); row = 2
for i in range(len(fd)):
    active = np.where(E_[i] > 1e-6)[0]
    for j in active:
        ws.cell(row,1).value = fd[i].strftime("%Y/%m/%d")
        ws.cell(row,2).value = f"{j*10//60}:{j*10%60:02d}-{(j+1)*10//60}:{(j+1)*10%60:02d}"
        ws.cell(row,3).value = float(E_[i,j]); row += 1
export_path = Path.cwd() / "result3_filled.xlsx"
wb.save(export_path)
print(f"已保存：{export_path}")

已保存：C:\Users\12055\Desktop\第三问\result3_filled.xlsx


## 7 结论

本版的问题3与问题2保持同一储能、SOC和实时结算口径，同时加入附件3及三次日内追索决策。新增参数全部在一月选择并于2月1日前冻结；最终费用、改进率和紧急购电指标以第3节当前运行表为准。

需要注意：分位数带是由单时段边际费用启发、再经一月滚动选择确定的阶段级规则。它具有清晰解释和较低计算量，但在含SOC耦合的多阶段模型中不应被表述为闭式严格解。